# `passage-eval` — Walkthrough 02: score released baselines from per-sample NPZ

The released vision and graph baselines emit per-sample NPZ artifacts.
This notebook scores them via `passage_eval.io.predictions` and emits
one `Submission` per (method, seed) into the same `results/`
leaderboard `01_astar_walkthrough.ipynb` already populated.

## 1. Imports

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PE_SRC = Path('~/projects/passage/passage/src/passage-eval/src')
if str(PE_SRC) not in sys.path: sys.path.insert(0, str(PE_SRC))
import passage_eval as pe

RESULTS_ROOT = Path('~/projects/passage/passage/src/passage-eval/results')
print('passage-eval version:', pe.__version__)


passage-eval version: 0.3.0


## 2. Vision baselines — load + score from `p1_a_per_sample/*.npz`

In [2]:
VISION_NPZ = Path('~/projects/passage/vision/outputs/paper/results/p1_a_per_sample')
SEEDS = [('primary', 1234), ('replica_a', 2026), ('replica_b', 3407)]

CONFIGS = [
    dict(label='seg-convnextv2_atto_baseline',
         family='dense_segmentation',
         architecture='convnextv2_atto',
         decoder_variant='BCE_threshold',
         template='segmentation_256x256_convnextv2_atto_elevation_baseline__{seed}__strict.npz',
         e2e_ms_p50=11.27),
    dict(label='reg-repvit_m1_bellman_vbridge',
         family='dense_regression',
         architecture='repvit_m1',
         decoder_variant='V_strict',  # released NPZ is V_strict; V_bridge is paper-Table-1
         template='regression_256x256_repvit_m1_elevation_bellman__{seed}__strict.npz',
         e2e_ms_p50=10.79),
]

for cfg in CONFIGS:
    for seed_name, seed_value in SEEDS:
        npz_path = VISION_NPZ / cfg['template'].format(seed=seed_name)
        arrays = pe.load_per_sample_npz(npz_path)
        # Apply the §5.2 censored penalty before metrics so failed decodes don't NaN the bootstrap.
        pred_censored = pe.apply_censored_penalty(arrays.pred_costs, arrays.oracle_costs,
                                                   decoder_succeeded=arrays.success)
        regret = np.where(np.isfinite(arrays.oracle_costs) & (arrays.oracle_costs > 0),
                           (pred_censored - arrays.oracle_costs) / arrays.oracle_costs, np.nan)
        # Re-derive a corrected PerSampleArrays bundle in-place (avoid mutating; build a small dict).
        metrics = pe.primary_metrics_from_arrays(
            pe.PerSampleArrays(
                regret=regret, assd=arrays.assd, h95=arrays.h95, success=arrays.success,
                pred_costs=pred_censored, oracle_costs=arrays.oracle_costs,
                obs_density=arrays.obs_density, elev_std=arrays.elev_std, oracle_len=arrays.oracle_len,
                extras=arrays.extras,
            )
        )
        sub = pe.Submission(
            method_name=cfg['label'], family=cfg['family'],
            architecture=cfg['architecture'], decoder_variant=cfg['decoder_variant'],
            resolution=256, cost_model='elevation', cost_weight=1.0,
            split='val', seed=seed_value, n_samples=metrics['n_samples'],
            metrics=metrics,
            edge_accelerator='Thor-GPU', batch_size=1, dtype='fp16',
            latency_e2e_ms_p50=cfg['e2e_ms_p50'],
            package_version=pe.__version__,
            dataset_version='Anonym-2045/passage:v0.4',
            extra={'notebook': '02_score_released_baselines',
                    'source_npz': str(npz_path.name)},
        )
        per_sample = {'regret': regret, 'assd': arrays.assd, 'h95': arrays.h95,
                      'success': arrays.success.astype(np.int8),
                      'pred_costs': pred_censored, 'oracle_costs': arrays.oracle_costs}
        pe.submit(sub, root=RESULTS_ROOT, per_sample=per_sample)
print('vision rows submitted')


vision rows submitted


## 3. Graph baseline — Edge-MLP from CSV (synthetic 64-sample smoke eval)

In [3]:
GRAPH_CSV = Path('~/projects/passage/graph/outputs/results/edge_mlp_eval_elevation.csv')
edge = pd.read_csv(GRAPH_CSV)
for _, row in edge.iterrows():
    metrics = {k: float(row[k]) for k in (
        'cost_regret_median','cost_regret_p90','path_assd','path_hausdorff_p95',
        'success_rate','obstacle_violation_rate',
    )}
    sub = pe.Submission(
        method_name='graph-edge_mlp_classical_astar',
        family='graph',
        architecture='edge_mlp_5_16_16_1',
        decoder_variant='analytic_a_star_on_learned_costs',
        resolution=int(row['resolution']),
        cost_model=str(row['cost_model']),
        cost_weight=1.0, split='val',
        seed=int(row['seed']), n_samples=int(row['n_samples']),
        metrics=metrics,
        edge_accelerator=None, batch_size=1, dtype='fp32',
        latency_fwd_ms_p50=float(row['latency_fwd_ms_p50']),
        latency_decoder_ms_p50=float(row['latency_dec_ms_p50']),
        latency_e2e_ms_p50=float(row['latency_total_ms_p50']),
        package_version=pe.__version__,
        dataset_version='passage_graph-synthetic-64',
        extra={'notebook':'02_score_released_baselines','source_csv':str(GRAPH_CSV.name)},
    )
    pe.submit(sub, root=RESULTS_ROOT)
print('edge-MLP rows submitted')


edge-MLP rows submitted


## 4. Read back + show all submissions

In [4]:
lb = pe.load_leaderboard(RESULTS_ROOT)
print('total rows:', len(lb))
display_cols = ['method_name','family','seed','resolution','cost_model',
                'cost_regret_median','cost_regret_p90','path_assd',
                'path_hausdorff_p95','success_rate','latency_e2e_ms_p50','n_samples']
lb.df[display_cols]


total rows: 308


,method_name,family,seed,resolution,cost_model,cost_regret_median,cost_regret_p90,path_assd,path_hausdorff_p95,success_rate,latency_e2e_ms_p50,n_samples
0,astar_classical,classical,1234,32,elevation,0.000000,0.000000,0.000000,0.000000,1.00000,0.500000,8
1,greedy_classical,classical,1234,32,elevation,0.087332,0.123439,0.028163,0.054626,1.00000,0.050000,8
2,seg-convnextv2_atto_baseline,dense_segmentation,1234,256,elevation,0.036936,0.142076,0.110557,0.196772,0.99707,11.270000,1024
3,seg-convnextv2_atto_baseline,dense_segmentation,2026,256,elevation,0.037426,0.142858,0.109595,0.194260,0.99707,11.270000,1024
4,seg-convnextv2_atto_baseline,dense_segmentation,3407,256,elevation,0.040326,0.145074,0.109070,0.193913,0.99707,11.270000,1024
...,...,...,...,...,...,...,...,...,...,...,...,...
303,reg-repvit_m1_bellman_vbridge,dense_regression,2026,256,elevation,0.001267,0.023260,0.022288,0.056379,0.99707,10.790000,1024
304,reg-repvit_m1_bellman_vbridge,dense_regression,3407,256,elevation,0.001018,0.020623,0.022701,0.057647,0.99707,10.790000,1024
305,graph-edge_mlp_classical_astar,graph,1234,256,elevation,0.113648,0.244158,0.082523,0.191006,1.00000,51.600044,64
306,graph-edge_mlp_classical_astar,graph,2026,256,elevation,0.139974,0.251914,0.061349,0.148888,1.00000,57.473581,64


All ten leaderboard rows (3 seeds × 2 vision + 3 seeds × 1 graph + 2 classical) live under
`results/runs/`; the next notebook visualises them.